In [ ]:
!pip install "gymnasium[atari]==1.3.0" "ale-py==0.11.2" "autorom[accept-rom-license]==0.6.1"
!AutoROM --accept-license

In [1]:
"""Environment Installation Verification Script.

Usage:
    python verify_env.py

Displays OK or FAIL for each step.
If all steps pass, you are ready to start training.
"""
import sys

TOTAL = 4

def step(idx, label):
    print(f"[{idx}/{TOTAL}] {label} ... ", end="", flush=True)

def ok(detail=""):
    print(f"OK  {detail}")

def fail(reason, hint=""):
    print("FAIL")
    print(f"    Cause: {reason}")
    if hint:
        print(f"    Fix: {hint}")
    sys.exit(1)

# ── 1. gymnasium import ─────────────────────────────────────────
step(1, "Gymnasium import")
try:
    import gymnasium
    ok(f"(version {gymnasium.__version__})")
    if not gymnasium.__version__.startswith("1.3"):
        print(f"    [Warning] Recommended version is 1.3.0. Current: {gymnasium.__version__}")
except Exception as e:
    fail(
        f"{type(e).__name__}: {e}",
        'pip install "gymnasium[atari]==1.3.0"',
    )

# ── 2. ale-py import + ALE registration ──────────────────────────
step(2, "ale-py import + ALE registration")
try:
    import ale_py
    if hasattr(gymnasium, "register_envs"):
        gymnasium.register_envs(ale_py)
    ok(f"(version {ale_py.__version__})")
except ImportError as e:
    fail(
        f"{e}",
        'pip install "ale-py==0.11.2" "autorom[accept-rom-license]==0.6.1" '
        "&& AutoROM --accept-license",
    )

# ── 3. ALE/Breakout-v5 environment creation ─────────────────────
step(3, "ALE/Breakout-v5 environment creation")
try:
    env = gymnasium.make(
        "ALE/Breakout-v5",
        frameskip=1,
        repeat_action_probability=0.0,
        full_action_space=False,
    )
    ok(f"(action_space={env.action_space})")
except gymnasium.error.NamespaceNotFound:
    fail(
        "Failed to register ALE namespace",
        "Try running 'AutoROM --accept-license' again",
    )
except Exception as e:
    fail(f"{type(e).__name__}: {e}")

# ── 4. Random play 100 steps ───────────────────────────────────
step(4, "Random play 100 steps")
try:
    obs, info = env.reset(seed=0)
    total_reward = 0.0
    for _ in range(100):
        action = env.action_space.sample()
        obs, reward, terminated, truncated, info = env.step(action)
        total_reward += float(reward)
        if terminated or truncated:
            obs, info = env.reset()
    env.close()
    ok(f"(Cumulative reward={total_reward})")
except Exception as e:
    fail(f"{type(e).__name__}: {e}")

print()
print("✅ All checks passed. You are ready to start training!")

[1/4] Gymnasium import ... OK  (version 1.3.0)
[2/4] ale-py import + ALE registration ... OK  (version 0.11.2)
[3/4] ALE/Breakout-v5 environment creation ... OK  (action_space=Discrete(4))
[4/4] Random play 100 steps ... OK  (Cumulative reward=0.0)

✅ All checks passed. You are ready to start training!
